#Classificação de Documentos com o modelo Bag-Of-Words (BOW)
Anteriormente, tivemos um exemplo de uso de abordagens de Classificação de Documentos para realizar a tarefa de Análise de Sentimentos. Neste material, vamos estudar diferentes estratégias utilizadas para melhorar a acurácia de tarefas de Classificação de Documentos.

Primeiramente, vamos fazer o download da base de dados de Análise de Sentimentos da plataforma Buscapé, de forma similar a aula passada.

In [1]:
import requests, os

url = "https://raw.githubusercontent.com/watinha/nlp-text-mining-datasets/main/buscape.csv"

def download (url):
  if (os.path.isfile('./buscape.csv')):
    print('Arquivo já existente no Runtime... Tudo OK')
    return
  response = requests.get(url)
  with open('./buscape.csv', 'wb') as f:
      f.write(response.content)
      print('Download realizado e arquivo extraído no Runtime... Tudo OK')

download(url)



Arquivo já existente no Runtime... Tudo OK


# Treinando e avaliando o modelo
Seguindo o mesmo código implementado na aula passada, para implementar um classifidor de sentimentos em comentários da plataforma Buscapé, vamos realizar os mesmos passos da aula anterior implementando:
1.   Carregar o dataset;
2.   Filtrar registros com dados faltantes (pré-processamento);
3.   Extrair os textos em um vetor de strings (corpus) e as polaridades em um vetor de rótulos (y);
4.   Definição dos grupos de treinamento e teste;
5.   Extração de características dos textos do corpus de treinamento;
6.   Treinar um modelo de aprendizado;
7.   Extração de características dos textos do corpus de teste;
8.   Predição e análise das métricas de acurácia.

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('./buscape.csv').dropna()                               # (1 e 2)

corpus = df['review_text'].tolist()                                      # (3)
y = df['polarity'].tolist()

corpus_train, corpus_test, y_train, y_test = train_test_split(corpus, y) # (4)

extractor = CountVectorizer(binary=True)                                 # (5)
X_train = extractor.fit_transform(corpus_train)

model = DecisionTreeClassifier(random_state=42)                          # (6)
model.fit(X_train, y_train)

X_test = extractor.transform(corpus_test)                                # (7)

y_pred = model.predict(X_test)                                           # (8)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.53      0.46      0.49      1737
         1.0       0.94      0.96      0.95     16670

    accuracy                           0.91     18407
   macro avg       0.74      0.71      0.72     18407
weighted avg       0.91      0.91      0.91     18407



Considerando o grupo de teste, foi observado que nosso classificador teve uma acurácia geral de 0,91, com F-Score para comentários com polaridade positiva de 0,95 e 0,49 para polaridade negativa. Nessa aula, vamos analisar estratégias de PLN e Mineração de Texto que podem ser utilizadas com o objetivo de melhorar a acurácia desses modelos, incrementando a taxa de acerto do nosso classificador.

Para facilitar as avaliações de acurácia do modelo, vamos implementar uma função para agrupar as chamadas de treinamento e avaliação do nosso modelo. Observem que a chamada dessa função, a princípio gera o mesmo resultado anterior.

In [3]:
def train_validate (corpus_train, corpus_test, y_train,
                    y_test, extractor, classifier):
  X_train = extractor.fit_transform(corpus_train)
  classifier.fit(X_train, y_train)

  X_test = extractor.transform(corpus_test)
  y_pred = classifier.predict(X_test)

  print(classification_report(y_test, y_pred))


extractor = CountVectorizer(binary=True)
classifier = DecisionTreeClassifier(random_state=42)

train_validate(corpus_train, corpus_test, y_train,
               y_test, extractor, classifier)

              precision    recall  f1-score   support

         0.0       0.53      0.46      0.49      1737
         1.0       0.94      0.96      0.95     16670

    accuracy                           0.91     18407
   macro avg       0.74      0.71      0.72     18407
weighted avg       0.91      0.91      0.91     18407



# Filtragem de stop-words
Uma estratégia frequentemente utilizada para reduzir a quantidade de dados de entrada dos modelos de classificação e, possivelmente, melhorar a acurácia dos modelos é a Filtragem de stop-words. Stop-words são palavras que apresentam uma alta frequência no texto, mas possuem baixo valor semântico (significado). Essas palavras geralmente são preposições, conjunções, artigos e outros termos comuns que não contribuem significativamente para a compreensão de textos.

Em aulas anteriores, nós exploramos o uso da biblioteca Spacy para executar diferentes tarefas de Processamento de Língua Natural. Entre as funcionalidades dessa biblioteca, ela disponibiliza uma lista de stop-words para a língua portuguesa.

```sh
!pip install spacy
!python -m spacy download pt_core_news_sm
```

In [4]:
from spacy.lang.pt import stop_words
print(stop_words.STOP_WORDS)

{'todos', 'nossa', 'está', 'faz', 'que', 'qualquer', 'final', 'então', 'meses', 'perto', 'falta', 'seu', 'pelo', 'aquelas', 'comprido', 'tive', 'veja', 'ainda', 'irá', 'pelas', 'tiveste', 'comprida', 'estiveram', 'entre', 'ambas', 'talvez', 'estar', 'saber', 'duas', 'bastante', 'tão', 'tua', 'terceiro', 'sobre', 'como', 'em', 'vosso', 'sempre', 'deverá', 'ou', 'esteve', 'posso', 'tentaram', 'eles', 'conhecida', 'treze', 'menos', 'contra', 'és', 'parece', 'outra', 'todo', 'vais', 'estão', 'tentar', 'isso', 'no', 'para', 'dezassete', 'nas', 'pontos', 'diz', 'do', 'demais', 'vários', 'sistema', 'números', 'cuja', 'nosso', 'embora', 'tiveram', 'segundo', 'não', 'apontar', 'foram', 'pode', 'certamente', 'põe', 'quê', 'ali', 'tendes', 'eventual', 'este', 'cá', 'dos', 'vai', 'cujo', 'me', 'outros', 'agora', 'vos', 'aos', 'enquanto', 'atrás', 'oito', 'pois', 'sabe', 'grupo', 'te', 'conhecido', 'muitos', 'vossos', 'fazemos', 'grandes', 'acerca', 'onze', 'naquela', 'quarta', 'teu', 'quinta', 'fo

Com essa lista disponibilizada pela biblioteca Spacy, torna-se possível implementar mecanismos para extrair essas palavras do corpus do nosso estudo. A API Vectorizer (CountVectorizer e TfidfVectorizer) também disponibiliza um parâmetro para remoção de stop-words do corpus que pode ser utilizada juntamente com a lista disponibilizada pela biblioteca Spacy.

In [5]:
extractor_2 = CountVectorizer(binary=True, stop_words=list(stop_words.STOP_WORDS))

X_train = extractor.fit_transform(corpus_train)
X_train_2 = extractor_2.fit_transform(corpus_train)
print(X_train.shape)
print(extractor.vocabulary_)
print(len(extractor.vocabulary_))
print(X_train_2.shape)
print(extractor_2.vocabulary_)
print(len(extractor_2.vocabulary_))

(55219, 49430)
{'produto': 37814, 'pequeno': 35405, 'mas': 30336, 'muito': 31972, 'prático': 38330, 'bom': 8010, 'fácil': 23524, 'de': 13808, 'mexer': 30966, 'coloca': 10636, 'nomes': 33042, 'ruas': 41692, 'bairros': 7049, 'cidades': 10143, 'enfim': 18612, 'recomendo': 39770, 'excelente': 20749, 'pois': 36427, 'eficiente': 17674, 'não': 33326, 'deixando': 14195, 'nada': 32374, 'desejar': 14991, 'com': 10696, 'relação': 40311, 'aos': 4867, 'fornos': 22761, 'gás': 24705, 'convencionais': 12546, 'mais': 29807, 'bonito': 8082, 'compacto': 10940, 'do': 16849, 'que': 38850, 'esses': 20049, 'gostei': 24278, 'funcional': 23303, 'dificil': 16067, 'limpar': 29050, 'achei': 2728, 'agradavel': 3635, 'só': 44724, 'ao': 4852, 'sei': 42545, 'se': 42404, 'perfume': 35569, 'continua': 12370, 'até': 6582, 'frasco': 22996, 'acabar': 2481, 'perfuma': 35560, 'como': 10909, 'um': 46929, 'banho': 7184, 'recem': 39600, 'tomado': 45758, 'custo': 13460, 'razoavel': 39363, 'ainda': 3783, 'colonia': 10665, 'está'

Nesse exemplo, observem que instanciamos a variável extractor_2 com a nova estratégia de tipo Vectorizer. No exemplo, apresentamos as dimensões da matriz X_train gerada pelo extrator com e sem a remoção das stop_words. Observem que a matriz X_train possui 49.417 colunas que representam os termos armazenados no dicionário de termos do extrator.

Em seguida, apresentamos as mesmas dimensões da matriz X_train_2 gerada pelo nosso segundo extrator. Dessa vez, a matriz X_train_2 possui 49.030 colunas (com 387 colunas a menos).

É importante destacar que nesse exemplo, apesar de diminuirmos a quantidade de dados processada pela nossa estratégia de Aprendizado Supervisionado, a acurácia foi similar.

In [6]:
train_validate(corpus_train, corpus_test, y_train, y_test, extractor_2, classifier)

              precision    recall  f1-score   support

         0.0       0.53      0.48      0.50      1737
         1.0       0.95      0.95      0.95     16670

    accuracy                           0.91     18407
   macro avg       0.74      0.72      0.73     18407
weighted avg       0.91      0.91      0.91     18407



#Pré-processamento e Lematização
Uma outra estratégia utilizada para aumentar a acurácia de estratégias de Classificação de Documentos é a Lematização. O objetivo dessa estratégia é similar à remoção de stop-words do texto, reduzindo as dimensões dos dados de entrada nos preditores. No entanto, ao invés de filtrar termos, a Lematização permite agrupar palavras que possuem a mesma raiz e, potencialmente, significado (por exemplo: amar, amor, amei, entre outros).

A tarefa de Lematização pode ser executada como etapa de Pré-processamento dos dados, trocando, no corpus, as palavras por seus lemas ou raizes.

In [7]:
# Nota: esse código demora 10 minutos para ser executado. Use com responsabilidade... :)
#   O tempo de demora é um resultado da realização de tarefas de PLN (tokenização e lematização)
#   em todas as amostras do dataset (84.991), considerando ambos os grupos (treinamento e teste).
#   Nesse exemplo, nós também limitamos o uso das tarefas de PLN do spacy, para tentar
#   reduzir o seu tempo de processamento, desabilitando os módulos de morphologizer,
#   senter, attribute_ruler e ner, que não serão utilizados no momento.
import spacy

pln = spacy.load("pt_core_news_sm", disable=["morphologizer", "senter", "attribute_ruler", "ner"])

def preprocessing (corpus):
  corpus_with_lemmas = []

  for text in corpus:
    doc = pln(text.lower())
    corpus_with_lemmas.append(' '.join([token.lemma_ for token in doc]))

  return corpus_with_lemmas

corpus_train = preprocessing(corpus_train)
corpus_test = preprocessing(corpus_test)

print(corpus_train[0:4])


['produto pequeno mas muito prático .', 'o produto ser muito bom fácil de mexer , colocar nome de rua , bairro , cidade ... \n enfim recomendo !', 'o produto ser excelente . recomendo , pois ser muito eficiente , não deixar nada a desejar com relação a o forno a gás convencional e ser muito mais bonito e compacto de o que esse . \n\n o que gostar : funcional , compacto e eficiente \n\n o que não gostar : dificil de limpar', 'achar muito bom , muito agradavel , só n´~ao saber se o perfume continuar até o frasco acabar \n\n o que gostar : perfuma como um banho recer tomar , e , o custo ser razoavel \n\n o que não gostar : ainda não saber , pois a o colonia ainda estar manter o perfume , ir observar atáo final de o vidro']


O código anterior executa um pipeline de tarefas de PLN para trocar as palavras do corpus por lemas. A saída do script anterior apresenta como ficaram os primeiros 5 documentos do grupo de treinamento.

Para ilustrar os resultados do uso da tarefa de Lematização, vamos executar o nosso extrator de características no corpus de treinamento com os lemas.

In [8]:
X_train_2 = extractor_2.fit_transform(corpus_train)
print(X_train_2.shape)
print(extractor_2.vocabulary_)
print(len(extractor_2.vocabulary_))

(55219, 40811)
{'produto': 31125, 'pequeno': 29104, 'prático': 31513, 'fácil': 18946, 'mexer': 25348, 'colocar': 8793, 'nome': 27144, 'rua': 34170, 'bairro': 5698, 'cidade': 8394, 'enfim': 15014, 'recomendo': 32722, 'excelente': 16653, 'eficiente': 14239, 'deixar': 11507, 'desejar': 12086, 'forno': 18272, 'gás': 20003, 'convencional': 10186, 'bonito': 6621, 'compacto': 9003, 'gostar': 19617, 'funcional': 18751, 'dificil': 12920, 'limpar': 23666, 'achar': 2337, 'agradavel': 3015, 'perfume': 29246, 'continuar': 10069, 'frasco': 18488, 'acabar': 2168, 'perfuma': 29238, 'banho': 5804, 'recer': 32604, 'tomar': 37641, 'custo': 10906, 'razoavel': 32380, 'colonia': 8803, 'manter': 24541, 'observar': 27501, 'atáo': 5308, 'vidro': 39542, 'philip': 29530, 'tv': 38409, 'excessão': 16696, 'moderno': 25750, 'desempenho': 12116, 'excepcional': 16685, 'marca': 24719, 'leve': 23489, 'manuseio': 24584, 'seca': 34833, 'rapidamente': 32286, 'cabelo': 7148, 'ajudar': 3174, 'controlar': 10145, 'frizz': 1859

O que pode ser observado é que a dimensionalidade do nosso problema de classificação foi reduzida novamente. A nossa matriz X_train_2 apresenta 42.703 colunas (anteriormente ela possuía 49.030). Essa redução é um resultado da aplicação da tarefa de Lematização, onde palavras diferentes com mesma raiz/lema foram agrupas em um mesmo termo dentro do nosso dicionário de termos.

Em seguida, vamos utilizar esse novo corpus para treinar e avaliar a acurácia do nosso modelo de Classificação de Sentimentos.

In [9]:
train_validate(corpus_train, corpus_test, y_train, y_test, extractor_2, classifier)

              precision    recall  f1-score   support

         0.0       0.51      0.47      0.49      1737
         1.0       0.95      0.95      0.95     16670

    accuracy                           0.91     18407
   macro avg       0.73      0.71      0.72     18407
weighted avg       0.90      0.91      0.91     18407



Nesse exemplo, embora não tenhamos ganho de acurácia, é importante observar que a complexidade do modelo geral foi reduzida (menos features), reduzindo as chances de problemas associados à variância/overfitting. E mesmo com o modelo simplificado, a acurácia geral da abordagem foi similar.

##Redução de Dimensionalidade utilizando NLTK
Nos exemplos anteriores, nós utilizamos a biblioteca SPACY para reduzir a dimensionalidade dos dados do nosso modelo de classificação, utilizando a filtragem de stopwords e lematização de palavras.

A remoção das stopwords foi realizada pela nossa própria estratégia de extração de características, com a API Count/Tfidf Vectorizer. No entanto, ela poderia ser realizada na etapa de preprocessamento, juntamente com a lematização.

Um outro ponto a ser observado está relacionado ao tempo de processamento necessário para a realização da lematização. Na etapa de lematização, é necessária a execução de múltiplas tarefas de PLN para que o lema seja identificado. A execução dessas tarefas em um córpus de tamanho médio para grande, demanda um tempo considerável de processamento.

Em determinados cenários, pode ser interessante utilizar uma estratégia Steming, en contraste à lematização. O Steming utiliza regras de redução de sufixos e prefixos para identificar a raiz da palavra. O Steming pode ser entendido como uma abordagem heurística para determinação das raizes das palavras e é muito mais rápida para ser executada em comparação à lematização.

A biblioteca SPACY não implementa estratégias de Steming. Para utilizar o Steming, nós vamos utilizar o NLTK, uma biblioteca de PLN, assim como o SPACY.

Para inicializar o NLTK, nós precisamos fazer o download das stopwords e da estratégia de Steming que será utilizada. Nesse exemplo, nós vamos utilizar a estratégia baseada no algoritmo RSLP.

In [10]:
import nltk

nltk.download('stopwords')
nltk.download('rslp')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\williamn\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.
[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\williamn\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping stemmers\rslp.zip.


True

Para utilizar o Steming, nós precisamos instanciar o Stemer e executar o método .stem para gerar a raiz das palavras.

In [11]:
stemer = nltk.stem.RSLPStemmer()

words = ['andando', 'passando', 'preambulo', 'navegando', 'adoção', 'adoçante', 'somos', 'sou']

[stemer.stem(word) for word in words]

['and', 'pass', 'preambul', 'naveg', 'adoç', 'adoç', 'som', 'sou']

É imporante destacar que o Steming pode gerar raizes de palavras com uma acurácio inferior à lematização. Em contrapartida, a tarefa de Steming pode ser executada muito mais rapidamente.

In [13]:
words = ['somos', 'fui', 'tiveste', 'teremos', 'direis', 'quis', 'pudemos']

lemmas_and_stemed = []

for token in pln(' '.join(words)):
  lemma = token.lemma_
  stemed = stemer.stem(token.text)
  print(f' - {token.text:20} - {lemma:20} - {stemed:20}')



 - somos                - ser                  - som                 
 - fui                  - ser                  - fui                 
 - tiveste              - tiveste              - tiv                 
 - teremos              - ter                  - ter                 
 - direis               - direl                - direl               
 - quis                 - quis                 - quil                
 - pudemos              - puder                - pud                 


Em seguida, vamos implementar uma estratégia de pré-processamento do texto baseada no uso do NLTK, para remoção de stopwords e Steming das palavras. Esse código executa uma tarefa similar ao código anterior de pré-processamento, mas vocês vão observar que sua execução exige bem menos tempo de processamento.

In [ ]:
def preprocessing (corpus):
  corpus_filtered = []
  stemer = nltk.stem.RSLPStemmer()
  stopwords = nltk.corpus.stopwords.words('portuguese')

  for row in corpus:
    tokens = row.split()
    tokens_filtered = [stemer.stem(token) for token in tokens
                       if token not in stopwords]
    corpus_filtered.append(' '.join(tokens_filtered))

  return corpus_filtered

X_train_filtered = preprocessing(corpus_train)
X_test_filtered = preprocessing(corpus_test)

#N-Gramas
O modelo Bag-Of-Words (BOW) considera a presença (CountVectorizer com a opção binary=True, por exemplo) ou frequência (TfidfVectorizer, por exemplo) de palavras para representar as informações de um documento em uma estrutura de dados. O modelo desconsidera a ordem das palavras no texto.

Existem cenários da Classificação de Documentos em que é interessante representar no modelo a relação entre termos e palavras. O modelo BOW pode representar essas relações por meio do conceito de N-Gramas.

N-Gramas são sequências contíguas de n itens de uma amostra de texto ou fala. Geralmente, "n" representa o número de itens. Por exemplo, em N-gramas de palavras (também chamados de "unigramas", "bigramas", "trigramas", etc.), um unigrama seria uma única palavra, um bigrama seria uma sequência de duas palavras consecutivas, e assim por diante. N-gramas capturam informações sobre a estrutura e a ordem das palavras em um texto.

Considerando como exemplo o seguinte texto:
> Esse produto foi pouco útil.

No cenário de Análise de Sentimentos, se utilizarmos apenas a extração de características com base em palavras, a presença/frequência das palavras pouco e útil seriam registradas individualmente. O modelo de classificação poderia considerar que o produto foi útil, por existir a presença desse termo. No entanto, se considerarmos o bigrama "pouco útil", ou seja, as duas palavras em conjunto, esse termo poderia ser associado a um sentimento negativo.

A extração de características de textos com n-gramas já é implementada nas classes de tipo Vectorizer (CountVectorizer e TfidfVectorizer), por meio do parâmetro ngram_range.






In [18]:
extractor_3 = CountVectorizer(binary=True, ngram_range=(1,3))
X_train_3 = extractor_3.fit_transform(corpus_train)
print(X_train_3.shape)
print(len(extractor_3.vocabulary_))

(55219, 1575252)
1575252


O parâmetro ngram_range é determinado com uma tupla contendo 2 valores que indicam a tamanho mínimo e máximo dos ngramas. No caso do código apresentado anteriormente, a extração de características será realizada com ngramas de tamanho 1, 2 e 3.

É importante também observar que a utilização de ngramas aumenta a complexidade do nosso modelo de classificação. Agora, a entrada do nosso modelo conta com 1.616.716 características.

A seguir, vamos analisar os efeitos do uso dessa abordagem no processo geral de classificação de sentimentos:

In [19]:
train_validate(corpus_train, corpus_test, y_train, y_test, extractor_3, classifier)

              precision    recall  f1-score   support

         0.0       0.63      0.51      0.56      1737
         1.0       0.95      0.97      0.96     16670

    accuracy                           0.93     18407
   macro avg       0.79      0.74      0.76     18407
weighted avg       0.92      0.93      0.92     18407



#Considerações sobre uso de BOW para Classificação de Documentos
O uso de estratégias como Filtragem de Stop-Words, Lematização e N-Gramas pode melhorar o processo de Classificação de Documentos. É importante observar que enquanto as estratégias de Filtragem de Stop-Words e Lematização reduzem a complexidade do modelo (diminuindo o número de características que serão analisadas pelos preditores), o uso de N-Gramas aumenta a complexidade do modelo.

Independemente, essas estratégias podem ser empregadas separadamente para tentar melhorar a acurácia dos modelos de Classificação de Documentos.

É importante também destacar que as aulas que abordagem o uso do modelo BOW para Classificação de Documentos foram realizadas na temática de Análise de Sentimentos. Mas existem diferentes cenários em que esse modelo poderia ser empregado. Existem diferentes estudos que exploram múltiplas tarefas que podem fazer uso dessa abordagem.

Para tentar melhorar a acurácia do modelo também podem ser utilizadas as mesmas técnicas exploradas anteriormente nesse curso e em outros cursos também, como:
* TfidfVectorizer;
* Algorítmos como LogisticRegression, SVM, RandomForest;
* GridSearchCV para definição dos hiper parâmetros;
* SelectKBest para seleção de características;
* Feature Scaling;
* Pipeline para melhorar a organização do código;
* entre outras.

A seguir, apresento um código que reporta o uso das abordagens mencionadas:
* [main.py](https://github.com/watinha/nlp-text-mining-datasets/blob/main/main.py)

